# 4.0.2 — plain 2-state **Gaussian** HMM on whisker ME

No AR filter. The hypothesis: **whisking is high or low**, and that level *is* the latent
state — the within-state dynamics may differ, but you do not need an autoregressive
emission to find the state.

Why this is worth a look rather than another AR variant:

| | AR-HMM | Gaussian HMM |
|---|---|---|
| lag hyperparameter | a grid, a cap, paired-vs-unpaired tests, bits-vs-raw | **none** |
| initialisation | `prior` only for Poisson; `kmeans` needs `emissions=` | **`kmeans` available** — and a 2-state high/low split is exactly what kmeans finds |
| what a longer filter buys | held-out LL keeps rising, but the segmentation converges by lag ~16 and duration drifts off the changepoint anchor | n/a |

So this is the lag-0 limit, and it removes the whole model-selection problem rather than
solving it. The question is only whether the **segmentation** is as good.

Fitting lives in `hmm_gaussian_functions.py`; it mirrors
`cross_validate_poismodel` and saves the **same pickle layout** as the AR pipeline (with
the lag fields neutral), so `4.0.1_hmm_dynamic_inspect.ipynb` and `hmm_dynamic_plots`
work on these fits unchanged.

In [ ]:
""" IMPORTS """
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import sys
# these notebooks now live in hmm_diagnostics/intermediate_pipeline/, so the
# shared segmentation_functions.py is two levels up
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

import hmm_gaussian_functions as G
import hmm_dynamic_plots as P
from hmm_dynamic_functions import load_fit_variable, prepare_batches, dwell_times
from segmentation_functions import idxs_from_files

%matplotlib inline

## Configuration

In [ ]:
# ==========================================================================
#                              RUN CODE
# ==========================================================================

prefix = '/home/ines/repositories/representation_learning_variability/'

data_path = prefix + 'paper-individuality/data/design_matrices/'
fps = 60.0
var_interest = ['whisker_me']

# ---- FITTING PARAMETERS ------------------------------------------------------
num_states = 2
num_train_batches = 5
method = 'kmeans'      # available here, unlike the Poisson model
fit_method = 'em'
num_iters = 100
kappa = 0.0            # unchanged from the AR work: the segmentation is invariant to it
zsc = True             # whisker ME is z-scored, as in 4.1

fitting_params = f'{num_train_batches}_{method}_{fit_method}_zsc_{zsc}_gaussian/'
save_path = prefix + 'paper-individuality/data/hmm/grid_search_gaussian/' + fitting_params
states_save_path = prefix + 'paper-individuality/data/hmm/most_likely_states/' + fitting_params
csv_path = os.path.join(save_path, f'assessments_{var_interest[0]}.csv')

# the AR fits, for comparison
ar_path = prefix + 'paper-individuality/data/hmm/grid_search_dynamic/5_prior_em_zsc_True_dynamic/'

os.makedirs(save_path, exist_ok=True)
print('save_path:', save_path)

## Pick a few sessions

Start small — a Gaussian fit is ~25 s per session, so a handful is a couple of minutes.
Set `EIDS = None` to run the whole cohort instead.

In [ ]:
EIDS = ['63f3dbc1', '034e726f', 'a8a8af78', '02fbb6da', '510b1a50', '15763234']
n_jobs = 4

all_files = os.listdir(data_path)
design_matrices = [i for i in all_files if 'design_matrix' in i and 'standardized' not in i]
idxs, mouse_names = idxs_from_files(design_matrices)

if EIDS is not None:
    idxs = [m for m in idxs if m[:36][:8] in EIDS]
print(f'{len(idxs)} sessions selected')

## Fit

In [ ]:
assessments = G.run_all_gaussian(
    idxs, var_interest, zsc, num_states, num_train_batches, method, fit_method,
    save_path=save_path, data_path=data_path, fps=fps,
    n_jobs=n_jobs, csv_path=csv_path, states_save_path=states_save_path,
    num_iters=num_iters, kappa=kappa)

cols = [c for c in ['mouse','eid','n_frames','n_segments','median_dwell_ms',
                    'occupancy_state1','mean_low','mean_high','raw_ll_per_frame',
                    'fit_ok','error'] if c in assessments]
assessments[cols]

## What does it look like?

`hmm_dynamic_plots` works on these fits directly, because the pickle layout matches. The
lag-profile panel is empty by design — there is no lag — so ask for the states panel only.

In [ ]:
win_s = 15
start_s = None      # None = auto-pick an informative window; or a time in seconds

for mouse, session in P.available_sessions(save_path, var_interest):
    fig, d = P.plot_session(save_path, data_path, var_interest,
                            mouse_name=mouse, session=session,
                            zsc=zsc, num_train_batches=num_train_batches, fps=fps,
                            win_s=win_s, start_s=start_s, panels=('states',),
                            figsize=(12, 3.0))
    plt.show()
    print(f"   state means (z): {np.round(d['emission_means'], 3)}")

## Gaussian vs AR, same sessions

Two cautions on the comparison:

- **Raw held-out LL is comparable, `bits_LL` is not.** Both models give log densities of
  the same observations, but a prior-sampled baseline differs between emission families —
  the same trap that made Poisson look like it beat Bernoulli.
- The AR model **conditions on past observations**, so it has strictly more information and
  should win on likelihood. That is not the question. The question is whether the
  *segmentation* and the *syllable durations* are as good, since those are what the
  downstream analysis uses.

In [ ]:
rows = []
for mouse, session in P.available_sessions(save_path, var_interest):
    g = P.load_result(save_path, var_interest, mouse, session)
    try:
        a = P.load_result(ar_path, var_interest, mouse, session)
    except FileNotFoundError:
        continue
    sg, sa = np.asarray(g['most_likely_states']), np.asarray(a['most_likely_states'])
    n = min(len(sg), len(sa))       # same data prep, so these should already match
    dg, da = dwell_times(sg), dwell_times(sa)
    rows.append(dict(mouse=mouse, eid=session[:8], ar_lag=a['best_lag'],
                     ar_dwell=float(np.median(da)) * 1000 / fps,
                     gauss_dwell=float(np.median(dg)) * 1000 / fps,
                     ar_nseg=len(da), gauss_nseg=len(dg),
                     ar_occ=float(np.mean(sa == 1)), gauss_occ=float(np.mean(sg == 1)),
                     ar_raw_ll=float(np.nanmean(a['all_lls'][a['best_lag']])),
                     gauss_raw_ll=float(np.nanmean(g['all_lls'][0])),
                     agree=float(np.mean(sg[:n] == sa[:n]))))
cmp = pd.DataFrame(rows)
display(cmp)

if len(cmp):
    print(f'median frame agreement       : {cmp.agree.median():.4f}')
    print(f'median dwell  AR {cmp.ar_dwell.median():.0f} ms  vs  Gaussian {cmp.gauss_dwell.median():.0f} ms')
    print(f'   (model-free changepoint anchor: 450 ms, IQR 379-550)')
    print(f'median raw held-out LL  AR {cmp.ar_raw_ll.median():+.4f}  vs  Gaussian {cmp.gauss_raw_ll.median():+.4f}')

## Side by side on the trace

The AR states as the reference ribbon, the Gaussian states overlaid, disagreements shaded —
the same comparison machinery as `compare_lag`, but across model families.

In [ ]:
for mouse, session in P.available_sessions(save_path, var_interest)[:6]:
    try:
        a = P.load_result(ar_path, var_interest, mouse, session)
    except FileNotFoundError:
        continue
    g = P.load_result(save_path, var_interest, mouse, session)
    sa = np.asarray(a['most_likely_states'])
    sg = np.asarray(g['most_likely_states'])[:len(sa)]

    sig = P.load_prepared(data_path, session, mouse, var_interest, zsc,
                          num_train_batches)[:len(sa), 0]
    fig, ax = plt.subplots(figsize=(12, 2.9))
    fig.patch.set_facecolor(P.SURFACE)
    P.plot_states(a, sig, fps, win_s=15, ax=ax, start_s=start_s,
                  compare_states=sg, labels=(f"AR lag {a['best_lag']}", 'Gaussian'))
    ax.set_title(f"{mouse}  {session[:8]}   ·   " + ax.get_title(), fontsize=9,
                 color=P.INK, loc='left', pad=17)
    plt.show()